In [1]:
import duckdb

# # Read-only (safe while no build is running)
# conn = duckdb.connect("data_manager/catalog_v2.duckdb", read_only=True)

# Read-only snapshot while a build IS running (copy first)
import shutil, os, time
snap = "/tmp/catalog_snap.duckdb"
for _ in range(60):
    try:
        shutil.copy2("../../data_manager/catalog_v2.duckdb", snap)
        conn = duckdb.connect(snap, read_only=True)
        break
    except duckdb.IOException:
        time.sleep(0.5)

In [2]:
df0 = conn.execute("""
    SELECT e.id, e.source, e.data_class, e.season, e.cluster, e.run,
           e.event_id, e.feature_hash, h.h5_path, h.part_key, h.local_idx
    FROM events AS e
    JOIN h5_locations h ON h.event_fk = e.id
    WHERE e.source=? AND e.data_class=? AND e.season=? AND e.cluster=? AND e.run=? AND e.event_id=?
""", ['exp_reco', 'exp_reco', 2020, 1, '1', 7908]).df()
df0.head()

,id,source,data_class,season,cluster,run,event_id,feature_hash,h5_path,part_key,local_idx
0,650030,exp_reco,exp_reco,2020,1,1,7908,579d5d84be699d4f,/home/albert/Baikal2025/data_manager/data/h5da...,part_s2020_c01_r0001,53


In [10]:
df = conn.execute("""
    SELECT *
    FROM events
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
df[df.source=='mc_merged'].head()

,id,source,data_class,season,cluster,run,event_id,feature_hash
15097842,15097861,mc_merged,muatm_2019,2019,3,part_1000,18,None
15097843,15097877,mc_merged,muatm_2019,2019,3,part_1000,34,None
15097844,15097983,mc_merged,muatm_2019,2019,2,part_1000,140,None
15097845,15098004,mc_merged,muatm_2019,2019,1,part_1000,161,None
15097846,15098101,mc_merged,muatm_2019,2019,1,part_1000,258,None


In [ ]:
import sys
sys.path.append("../..")
from data_manager.catalog_v2.retriever import EventCatalog

with EventCatalog(catalog_path="/tmp/catalog_snap.duckdb") as cat:
    # Summary dict: {(source, data_class): count}
    print(cat.summary())

    # Single event lookup
    ev = cat.get(source='mc_merged', season=2020, cluster=1, run='1', event_id=2718)
    hits = ev.load_hits()     # (n_hits, 5) float32  [amp, t, x, y, z]
    reco = ev.load_reco()     # dict of reco scalars (exp_reco only)
    print(ev.info)

    # Batch query → list[Event]
    events = cat.query(source='exp', season=2020, cluster=7)
    hits_list = [e.load_hits() for e in events[:10]]

    # Batch query → pandas DataFrame (metadata only)
    df = cat.query_df(source='mc_merged', data_class='nuatm_2020')

{('exp', 'exp'): 649976, ('mc_merged', 'muatm_2019'): 4539251, ('exp_reco', 'exp_reco'): 14447866, ('mc_merged', 'muatm_2020'): 94756128}
{'source': 'exp_reco', 'data_class': 'exp_reco', 'season': 2020, 'cluster': 1, 'run': '1', 'event_id': 2718}


In [7]:
events[0]

Event(id=549977, source='exp', data_class='exp', season=2020, cluster=7, run='41', event_id=0, feature_hash='ab8a6e929f81d208')

In [5]:
hits_list[0].shape

(106, 5)

In [12]:
conn.close()
os.unlink(snap)